# 📗 추론 가속: 서버 설정을 vllm bench sweep serve로 비교하고 정하기

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<div style="background:#ffffff;color:#202124;font-size:17px;line-height:1.6;padding:6px 10px;border-left:4px solid #54769b;">
<strong style="color:#1f3a5f;font-size:19px;">실습 환경</strong><br>
<strong style="font-size:1.13em;color:#1f3a5f;">서버:</strong> RunPod의 Linux + NVIDIA GPU에서 vLLM 0.27.0 서버와 벤치마크를 실행합니다. vLLM은 macOS·Windows 기본 환경에 설치되지 않습니다.<br>
<strong style="font-size:1.13em;color:#1f3a5f;">클라이언트:</strong> 내 PC(Mac·Windows·Linux)의 VS Code에서 노트북을 실행해 공개 주소로 요청하고 응답을 봅니다. 여러 결과의 분석도 내 PC에서 합니다. PC에는 GPU가 필요 없습니다.<br>
<strong style="font-size:1.13em;color:#1f3a5f;">준비:</strong> 지난 단원 <a style="color:#245fa0;text-decoration:underline;" href="../day75_vLLM_추론엔진_서빙/부록_01_RunPod_환경과_외부_API_연결.ipynb">부록 01</a>과 같게 Pod를 만들고(PyTorch 템플릿 CUDA 13.0, HTTP 8000 포트 공개), 같은 .env(VLLM_SERVER_URL, VLLM_API_KEY)를 씁니다.<br>
<strong style="font-size:1.13em;color:#1f3a5f;">최소 권장:</strong> GPU VRAM 24GB 1장 · RAM 16GB · 디스크 80GB. 서버는 하나만 실행합니다.
</div>
</div>

#### **추론 가속 서버 설정**

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">모델과 GPU는 그대로 두고 <strong style="font-size:1.13em;color:#1f3a5f;">서버 옵션만 바꿔, 같은 GPU에서 지연을 줄이거나 출력 처리량을 늘리는 설정</strong>입니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">배우는 이유:</strong> 서버를 늘리기 전에, 같은 GPU로 지연 목표를 지키며 처리하는 요청을 늘립니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">활용처:</strong> 참고 문서가 긴 RAG 서버와 상담 서버, 동시 요청이 몰리는 사내 챗봇 서버의 설정 결정입니다.</p>
</div>

<img style="display:block;max-width:100%;height:auto;margin:0;" src="images/16_acceleration_concept.png" width="1050" alt="같은 GPU 서버에서 접두어 캐싱, KV 캐시 자료형, 동시 실행 상한, 투기적 디코딩 설정을 바꾸고, 지연과 처리량을 비교해 설정을 고릅니다.">

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">교안 01의 노무 상담 서버로 추론 가속 설정 네 가지를 배웁니다. 공개 서버에서 접두어 캐싱의 캐시 적중을 확인한 뒤, 서버 설정마다 같은 벤치마크를 반복하는 sweep(스윕)을 RunPod에서 <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">vllm bench sweep serve</code> 명령 하나로 실행</strong>해 기준 설정과 MTP를 비교하고, 예상 동시 요청 수에서 지연 목표를 지키는 설정을 고릅니다.</p>
</div>

<img style="display:block;max-width:100%;height:auto;margin:0;" src="images/10_lesson02_goals.png" width="1050" alt="학습 목표 그림: 1 접두어 캐싱, 2 KV 캐시 자료형 fp8, 3 동시 실행 상한, 4 투기적 디코딩">

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">

**학습 목표** (그림의 번호와 같습니다)

1. 앞부분이 같은 요청에서 접두어 캐싱의 캐시 적중을 확인하고, 캐싱을 끈 서버와 켠 서버의 TTFT와 출력 처리량을 측정 예로 비교합니다.
2. `--kv-cache-dtype fp8`이 KV 캐시에 담는 토큰 수를 늘리는 원리와 쓰는 때를 설명합니다.
3. `--max-num-seqs`를 줄이면 TPOT는 짧아지고 TTFT와 출력 처리량은 나빠지는 이유를 설명합니다.
4. MTP 투기적 디코딩의 수용 길이와 동시 요청 수별 효과를 기준 설정과 `vllm bench sweep serve`로 한 번에 측정해 비교하고, 서버 설정을 정합니다.

</div>

## 1. 실습 준비

### 1-1. 내 PC 준비

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">아래 Python 셀은 모두 <strong style="font-size:1.13em;color:#1f3a5f;">내 PC</strong>에서 실행합니다. 교안 01의 준비 코드(경로, 설정값, 데이터, 토크나이저)와 공개 서버에 요청을 보낼 LangChain 채팅 모델 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">llm</code>을 한 셀에 모았습니다. 공개 서버 주소와 API 키는 교안 01과 같은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">.env</code>에서 읽습니다.</p>
<p style="margin:6px 0;">데이터는 교안 01과 같은 참고 조문 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">data/본실습/labor_reference.json</code>과 질문 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">data/본실습/labor_questions.jsonl</code>입니다. 벤치마크 입력 파일은 교안 01에서 만들어 RunPod에 올린 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">output/labor_prompts.jsonl</code>을 그대로 씁니다.</p>
</div>

In [ ]:
# ---- 라이브러리, 경로, 설정값 ----
import json
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv, find_dotenv
from transformers import AutoTokenizer

# 내 PC의 .env에서 서버의 공개 주소와 API 키를 읽습니다
load_dotenv(find_dotenv(usecwd=True))
material_dir = Path(".").resolve()  # 이 단원 폴더(정답 노트북은 "..")
data_dir = material_dir / "data"
output_dir = material_dir / "output"
bench_dir = output_dir / "bench"           # RunPod에서 내려받은 벤치마크 결과 JSON을 두는 폴더
sweep_dir = output_dir / "sweep"           # RunPod에서 내려받은 설정 비교 결과(summary.csv)를 두는 폴더
bench_dir.mkdir(parents=True, exist_ok=True)
sweep_dir.mkdir(parents=True, exist_ok=True)
SERVER_URL = os.environ["VLLM_SERVER_URL"].rstrip("/")  # RunPod 공개 HTTPS 주소
VLLM_API_KEY = os.environ["VLLM_API_KEY"]

# 설정값
MODEL_NAME = "Qwen/Qwen3.5-4B"            # 서버가 불러온 허브 모델(토크나이저도 이 이름으로 받습니다)
SERVED_NAME = "qwen3.5-4b"               # 서버 이름(--served-model-name). 요청의 model에 넣습니다
MAX_MODEL_LEN = 8192                  # 참고 조문이 든 시스템 메시지 + 질문 + 답을 담는 최대 길이
MAX_TOKENS = 256                     # 요청마다 생성 상한(지난 단원과 같은 값)
TTFT_TARGET_MS = 1000                # 지연 목표: 첫 토큰까지 1000ms 안
TPOT_TARGET_MS = 50                  # 지연 목표: 출력 토큰 하나에 50ms 안

# 테이블의 긴 답변이 잘리지 않게 칸의 글자 수 제한을 풉니다
pd.set_option("display.max_colwidth", None)

# 그래프 글꼴: 영문, 숫자, 기호는 DejaVu Sans, 한글은 운영체제의 한글 글꼴이 맡습니다
import platform

import matplotlib.pyplot as plt
from matplotlib import font_manager

if platform.system() == "Windows":
    korean_font = "Malgun Gothic"
elif platform.system() == "Darwin":
    korean_font = "AppleGothic"
else:
    korean_font = "NanumGothic"
    # 리눅스: 글꼴을 설치하기 전에 만든 matplotlib 글꼴 목록에는 나눔고딕이 없으므로, 설치된 파일을 직접 등록합니다
    for font_path in font_manager.findSystemFonts():
        if "NanumGothic" in font_path:
            font_manager.fontManager.addfont(font_path)
plt.rcParams["font.family"] = ["DejaVu Sans", korean_font]
plt.rcParams["axes.unicode_minus"] = False  # 음수 부호가 네모로 깨지지 않게 합니다

# 지난 단원과 같은 지침 문장. 시스템 메시지는 이 문장 뒤에 참고 조문을 붙여 만듭니다
SYSTEM_PROMPT = "너는 노무 상담 챗봇이다. 노동법 조문에 근거해 존댓말로 간결히 답한다."

# ---- JSONL 읽기(교안 01과 같은 함수) ----
def load_jsonl(path):
    """JSONL 파일을 한 줄에 딕셔너리 하나씩 읽어 리스트로 돌려줍니다(지난 단원과 같은 함수)."""
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return rows


# ---- 데이터, 토크나이저 ----
# 노무 상담의 참고 조문과 질문, 시스템 메시지 = 지침 문장 + 빈 줄 + [참고 조문] + 조문들
with open(data_dir / "본실습" / "labor_reference.json", encoding="utf-8") as f:
    labor_reference = json.load(f)
labor_questions = load_jsonl(data_dir / "본실습" / "labor_questions.jsonl")
labor_articles = [article["text"] for article in labor_reference["articles"]]
system_prompt = SYSTEM_PROMPT + "\n\n[참고 조문]\n" + "\n".join(labor_articles)

# HF_TOKEN 없이 내려받으면 'unauthenticated requests' 경고가 찍힙니다. 공개 모델이라 그대로 진행해도 됩니다
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# ---- 공개 서버에 요청할 채팅 모델(교안 01과 같습니다) ----
from langchain_openai import ChatOpenAI

# 공개 서버 주소와 API 키로 요청하는 LangChain 채팅 모델(지난 단원 ChatOpenAI 와 같은 모양)
llm = ChatOpenAI(
    model=SERVED_NAME,                 # 서버 이름(--served-model-name)
    base_url=f"{SERVER_URL}/v1",       # RunPod 공개 HTTPS 주소 + /v1
    api_key=VLLM_API_KEY,              # 서버를 띄울 때 정한 --api-key
    temperature=0,
    max_tokens=MAX_TOKENS,
    stream_usage=True,                 # base_url을 직접 정하면 기본값이 꺼짐이라, 스트리밍 응답에서도 토큰 사용량을 받으려고 켭니다
)

# 교안 01에서 만들어 RunPod에 올린 파일(조문이 앞, 질문이 뒤)
labor_prompts_path = output_dir / "labor_prompts.jsonl"
print("질문 수:", len(labor_questions), "| 시스템 메시지 토큰 수:", len(tokenizer.encode(system_prompt)), "| 입력 파일:", labor_prompts_path.name)

### 1-2. 공개 서버 확인

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">교안 01의 공개 서버가 떠 있으면 그대로 씁니다.</strong> RunPod 터미널 1에 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Application startup complete</code> 뒤로 오류 없이 로그가 이어지고 있으면 떠 있는 것입니다. 서버를 이어 쓰면 2절을 마칠 때까지 서버를 새로 띄우지 않습니다.</p>
<p style="margin:6px 0;">서버를 껐거나 파드를 새로 시작했다면 RunPod 터미널 1에서 아래 두 블록을 차례로 실행합니다. 파드를 새로 만들었다면 교안 01에서 올린 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">labor_prompts.jsonl</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">lease_prompts.jsonl</code>을 RunPod <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">output/</code>에 다시 올리고, 바뀐 공개 주소를 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">.env</code>의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">VLLM_SERVER_URL</code>에 고친 뒤 커널을 다시 시작합니다.</p>
</div>

**복사해서 실행 · RunPod 터미널 1**

```bash
# 실습자료 폴더에서 연 터미널에서 실행합니다
cd day76_추론가속_서빙벤치마킹
pwd   # 끝이 day76_추론가속_서빙벤치마킹이면 맞습니다
mkdir -p output/bench output/sweep
# 서버 API 키를 입력합니다. VLLM_API_KEY는 서버가 요청을 인증할 키, OPENAI_API_KEY는 vllm bench serve가 요청에 붙이는 키입니다
read -s -p "서버 API 키 입력: " VLLM_API_KEY && echo && export VLLM_API_KEY OPENAI_API_KEY="$VLLM_API_KEY"
```

**복사해서 실행 · RunPod 터미널 1**

```bash
uv run vllm serve Qwen/Qwen3.5-4B \
  --served-model-name qwen3.5-4b \
  --max-model-len 8192 \
  --language-model-only \
  --reasoning-parser qwen3 \
  --default-chat-template-kwargs '{"enable_thinking": false}' \
  --enable-prompt-tokens-details \
  --enable-prefix-caching \
  --host 0.0.0.0 \
  --port 8000 \
  --api-key "$VLLM_API_KEY" 2>&1 | tee output/qwen3.5-4b_server.log
```

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">로그에 <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">Application startup complete</code></strong>가 나오면 요청을 보낼 수 있습니다. 서버 출력은 RunPod의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">output/qwen3.5-4b_server.log</code>에도 남습니다. <strong style="font-size:1.13em;color:#1f3a5f;">RunPod 터미널 1</strong>은 그대로 두고 <strong style="font-size:1.13em;color:#1f3a5f;">내 PC의 노트북</strong>으로 돌아옵니다.</p>
<p style="margin:6px 0;">첫 실행은 모델 다운로드와 GPU 계산 준비에 시간이 걸릴 수 있습니다.</p>
</div>

## 2. 접두어 캐싱

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">네 설정은 요청 처리 단계 하나씩을 바꿉니다. 접두어 캐싱은 prefill, KV 캐시 자료형은 KV 캐시, 동시 실행 상한은 스케줄러, 투기적 디코딩은 decode를 바꿉니다.</p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">KV 캐시:</strong> 교안 01 1절에서 본, 앞 토큰들의 계산 결과(K·V)를 저장해 두는 GPU 메모리입니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">스케줄러:</strong> 생성 단계마다 어떤 요청을 함께 계산할지 고르는 vLLM의 부분입니다. 지난 단원 교안 01 5-4의 연속 배치가 여기서 일어납니다.</li>
</ul>
<img style="display:block;max-width:100%;height:auto;margin:0;" src="images/15_acceleration_settings.png" width="1050" alt="prefill, KV 캐시, 스케줄러, decode 순서의 요청 처리 흐름에 접두어 캐싱, KV 캐시 자료형, 동시 실행 상한, 투기적 디코딩이 각각 prefill, KV 캐시, 스케줄러, decode에 연결된다.">
</div>

### 2-1. 블록 단위로 다시 쓰는 KV 캐시

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">블록:</strong> KV 캐시를 정해진 토큰 수씩 나눠 담는 칸입니다(지난 단원 교안 01 5-3).</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">접두어 캐싱(Automatic Prefix Caching)</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">요청마다 같은 앞부분의 KV 캐시를 블록 단위로 남겨 두고, 다음 요청의 앞부분이 같으면 그 블록의 prefill을 건너뛰고 다시 쓰는 기능입니다. 서버 옵션은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--enable-prefix-caching</code>입니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">켜는 법:</strong> vLLM 0.27.0은 Qwen3.5처럼 attention 층과 Gated DeltaNet 층을 섞어 쓰는 하이브리드 모델에서 이 기능을 기본으로 끕니다. 그래서 교안 01의 서버 명령이 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--enable-prefix-caching</code>으로 켜고, 이 옵션이 없으면 이 모델은 캐싱 없이 돕니다. 하이브리드 모델의 접두어 캐싱은 vLLM이 아직 실험 기능으로 표시합니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">블록을 알아보는 방법:</strong> vLLM은 다 찬 블록마다, 앞 블록의 식별값과 이 블록의 토큰으로 계산한 짧은 식별값(해시)을 붙여 기억해 둡니다. 그래서 식별값이 같으면 맨 앞부터 그 블록까지 전부 같다는 뜻입니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">목적:</strong> 시스템 메시지, 참고 문서, 이전 대화처럼 요청마다 같은 앞부분(공통 접두어, common prefix)의 prefill을 줄여 TTFT를 줄이고 GPU 시간을 아낍니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">캐시 적중 토큰(cached tokens):</strong> 입력 토큰 가운데 캐시에서 다시 쓴 토큰 수입니다. 서버 명령의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--enable-prompt-tokens-details</code>가 이 값을 응답에 넣고, LangChain 응답에서는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">usage_metadata[&quot;input_token_details&quot;][&quot;cache_read&quot;]</code>로 읽습니다.</li>
</ul>
<img style="display:block;max-width:100%;height:auto;margin:0;" src="images/11_prefix_caching.png" width="1050" alt="시스템 메시지가 같은 둘째 요청은 꽉 찬 블록만 다시 쓰고, 덜 찬 마지막 블록의 시스템 메시지 끝부분과 질문은 새로 계산합니다. 질문을 앞에 두면 첫 블록부터 달라 다시 쓸 블록이 없습니다.">

<p style="margin:6px 0;">블록은 통째로 같아야 다시 씁니다. 블록 하나가 528토큰이고 입력이 교안 01에서 센 약 4,980토큰(시스템 메시지 약 4,930토큰)인 경우를 따라가 봅니다.</p>
<ol style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">시스템 메시지 가운데 앞 블록 9개(528 × 9 = 4,752토큰)가 꽉 찹니다.</li>
<li style="margin:3px 0;">같은 시스템 메시지의 둘째 요청은 이 블록 9개를 다시 쓰고, 블록을 채우지 못한 나머지 시스템 메시지 토큰과 질문만 prefill 합니다.</li>
<li style="margin:3px 0;">질문이 맨 앞에 오면 첫 블록부터 요청마다 내용이 달라 다시 쓸 블록이 없습니다.</li>
</ol>
<p style="margin:6px 0;">Qwen3.5는 지난 단원 교안 01 5-3에서 본 대로 두 종류의 층을 섞어 씁니다. attention 층은 토큰마다 KV를 쌓고, 선형 attention 층(Gated DeltaNet)은 요청마다 크기가 정해진 상태 하나만 저장합니다. vLLM은 attention 블록 하나가 이 상태 하나보다 작지 않도록 블록을 크게 잡습니다. 서버 시작 로그의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Setting attention block size to ... tokens</code> 줄이 그 크기이고, 캐시 적중은 이 블록 크기 단위로만 일어납니다.</p>
</div>

### 2-2. 캐시 적중 확인

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">공개 서버에 메시지 세 개를 차례로 보냅니다. 처음 둘은 같은 시스템 메시지(조문이 앞)에 서로 다른 질문이고, 셋째는 둘째 질문을 질문이 앞인 메시지(시스템 메시지는 지침 문장만, user 메시지는 질문 뒤에 조문)로 바꾼 것입니다.</p>
</div>

In [ ]:
# 보낼 메시지 세 개: (프롬프트 순서, 질문, 메시지)
cache_requests = [
    ("조문을 앞에", labor_questions[0]["question"], [("system", system_prompt), ("human", labor_questions[0]["question"])]),
    ("조문을 앞에", labor_questions[1]["question"], [("system", system_prompt), ("human", labor_questions[1]["question"])]),
    ("질문을 앞에", labor_questions[1]["question"], [("system", SYSTEM_PROMPT), ("human", labor_questions[1]["question"] + "\n\n[참고 조문]\n" + "\n".join(labor_articles))]),
]

# 차례로 보내고(스트리밍 없이 temperature=0으로), 입력 토큰 수와 그 가운데 캐시에서 다시 쓴 토큰 수를 모읍니다
cache_hit_rows = []
for order, question, messages in cache_requests:
    response = llm.invoke(messages)
    cache_hit_rows.append({
        "프롬프트 순서": order,
        "질문": question,
        "입력 토큰 수(input_tokens)": response.usage_metadata["input_tokens"],
        "캐시 적중 토큰(cache_read)": response.usage_metadata["input_token_details"]["cache_read"],
    })

cache_hit_table = pd.DataFrame(cache_hit_rows)
display(cache_hit_table)

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">터미널 1은 서버를 실행 중이므로 RunPod 터미널 2에서 블록 크기를 확인합니다.</p>
</div>

**복사해서 실행 · RunPod 터미널 2**

```bash
# 새 터미널이면 이 블록부터 실행합니다
# 실습자료 폴더에서 연 터미널에서 실행합니다
cd day76_추론가속_서빙벤치마킹
pwd   # 끝이 day76_추론가속_서빙벤치마킹이면 맞습니다
mkdir -p output/bench output/sweep
# 서버 API 키를 입력합니다. VLLM_API_KEY는 서버가 요청을 인증할 키, OPENAI_API_KEY는 vllm bench serve가 요청에 붙이는 키입니다
read -s -p "서버 API 키 입력: " VLLM_API_KEY && echo && export VLLM_API_KEY OPENAI_API_KEY="$VLLM_API_KEY"
```

**RunPod 터미널 2 · 화면에서 확인**

```bash
grep "attention block size" output/qwen3.5-4b_server.log
```

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">결과 읽기</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">둘째 행(조문을 앞에):</strong> 시스템 메시지가 같아 앞 블록들을 다시 씁니다. 캐시 적중 토큰은 화면의 블록 크기의 정수배이고, 나머지 시스템 메시지 끝부분과 질문만 새로 prefill 합니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">셋째 행(질문을 앞에):</strong> 같은 질문과 같은 조문이지만 첫 블록부터 앞의 요청과 내용이 달라 캐시 적중 토큰이 0입니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">첫 행:</strong> 교안 01의 서버를 이어 쓰면 같은 시스템 메시지를 이미 보냈으므로 첫 행도 적중할 수 있습니다. 그 사이 전월세, 온라인 쇼핑 벤치마크를 많이 돌렸다면 노무 블록이 다른 요청에 자리를 내주고 지워져 0일 수도 있습니다. 서버를 새로 띄웠다면 다시 쓸 블록이 없어 0입니다. 이 셀을 다시 실행하면 셋째 행도 앞 실행의 블록을 다시 써 적중합니다.</li>
</ul>
</div>

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">측정 예: 접두어 캐싱을 끈 서버와 켠 서버</strong></p>
<p style="margin:6px 0;">강사가 RunPod L4 24GB, vLLM 0.27.0에서 교안 01 4-2와 같은 조건(노무 입력, 동시 요청 16, 요청 64개)으로 접두어 캐싱만 끄고 켜서 측정한 예입니다. 직접 다시 측정하지 않으며, GPU가 다르면 숫자도 달라집니다.</p>
<table style="background:#ffffff;color:#202124;border-collapse:collapse;margin:6px 0;">
<thead>
<tr>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 8px;">항목</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 8px;">접두어 캐싱 끔</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 8px;">접두어 캐싱 켬</th>
</tr>
</thead>
<tbody>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">걸린 시간(초)</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">89.6</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">36.2</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">출력 처리량(토큰/초)</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">107.4</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">274.1</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">평균 TTFT(ms)</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">2,892</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">280</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">p95 TTFT(ms)</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">11,214</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">760</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">평균 TPOT(ms)</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">124.1</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">47.8</td>
</tr>
</tbody>
</table>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">결과 읽기</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">TTFT:</strong> 캐싱을 켜면 요청마다 시스템 메시지 앞부분 4,752토큰(528토큰 블록 9개)을 다시 써, 평균 TTFT가 2,892ms에서 280ms로 줄었습니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">TPOT:</strong> 캐싱을 끄면 새 요청마다 약 5,000토큰의 prefill이 생성 단계에 섞여, 이미 답을 만들던 요청도 느려집니다. 그래서 평균 TPOT도 124.1ms에서 47.8ms로 줄었습니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">비용:</strong> 같은 GPU가 같은 요청 64개를 약 2.5배 빨리 끝냈으므로, 토큰 하나에 드는 비용도 그만큼 줄어듭니다.</li>
</ul>
</div>

### 2-3. 공개 서버 끄기

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">공개 서버로 하는 실습은 여기까지입니다. 6절의 sweep이 같은 GPU와 포트 8000에 서버를 설정마다 차례로 띄우므로 지금 공개 서버를 끕니다. 3~5절은 개념 설명이라 서버가 필요 없습니다.</p>
</div>

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">RunPod 터미널 1</strong>(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">qwen3.5-4b</code> 서버를 띄운 터미널)에서 <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">Ctrl+C</code></strong>를 누릅니다. 명령을 입력할 수 있게 되면 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">nvidia-smi</code>를 실행합니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">vLLM 프로세스가 목록에서 사라지고 GPU 메모리 사용량이 줄었으면</strong> 서버가 꺼진 것입니다. 모델 가중치와 KV 캐시가 쓰던 메모리도 이때 반환됩니다.</p>
</div>

## 3. KV 캐시 자료형: --kv-cache-dtype

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">시작 로그의 KV 캐시 토큰 수가 클수록 긴 요청과 많은 동시 요청을 함께 담습니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">KV 캐시 자료형(KV cache dtype, <code style="color:inherit;background:#f3f6fa;font-size:0.95em;">--kv-cache-dtype</code>)</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">KV 캐시에 값을 저장하는 자료형입니다. 기본은 모델 자료형(bf16, 값 하나 2바이트)이고, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--kv-cache-dtype fp8</code>은 fp8(8-bit floating point, 8비트 부동소수, 값 하나 1바이트)로 저장해 같은 메모리에 토큰을 더 담습니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">목적:</strong> 긴 문맥이나 많은 동시 요청으로 KV 캐시가 모자랄 때 담을 수 있는 토큰 수를 늘립니다.</li>
</ul>
<img style="display:block;max-width:100%;height:auto;margin:0;" src="images/14_kv_cache_dtype.png" width="1050" alt="fp8은 attention KV를 값 하나 1바이트로 저장해 절반으로 줄이지만 Gated DeltaNet 상태는 그대로입니다. 그래서 같은 메모리에 담는 토큰은 2배가 아니라 약 1.5배입니다.">

<p style="margin:6px 0;">Qwen3.5는 attention 층의 KV만 1바이트로 줄고, Gated DeltaNet 층의 상태(자료형은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--mamba-cache-dtype</code>이 따로 정합니다)는 그대로입니다. 그래서 담는 토큰 수는 2배보다 적게 늘어납니다. 그리고 KV 캐시 자료형을 바꾸면 attention 블록 크기도 바뀔 수 있어, 접두어 캐싱의 적중 단위가 달라집니다. 저장하는 값의 정밀도가 낮아져 답이 조금 바뀔 수 있으므로, 답 품질은 SFT 단원의 판사 평가로 다시 확인합니다.</p>
<p style="margin:6px 0;">이 교안은 fp8 KV를 설명만 하고 측정하지 않습니다. 동시 요청이 많아 KV 캐시가 모자랄 때 기준 설정과 같은 부하로 측정해 비교합니다.</p>
</div>

## 4. 동시 실행 상한: --max-num-seqs

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">동시 실행 상한(<code style="color:inherit;background:#f3f6fa;font-size:0.95em;">--max-num-seqs</code>)</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">스케줄러가 한 생성 단계에 함께 계산하는 요청 수의 상한입니다. 상한을 넘은 요청은 서버의 대기열(queue)에서 자리가 날 때까지 기다립니다. 24GB GPU에서 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">vllm serve</code>의 기본값은 256입니다.</li>
<li style="margin:3px 0;">벤치마크의 동시 요청 수(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--max-concurrency</code>)는 보내는 쪽의 상한이고, 동시 실행 상한은 서버 쪽의 상한입니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">목적:</strong> 부하가 GPU 처리 능력을 넘을 때 TPOT가 길어지지 않게 지킵니다. 그 대가로 대기열에서 기다린 시간이 TTFT에 더해집니다.</li>
</ul>
<img style="display:block;max-width:100%;height:auto;margin:0;" src="images/12_max_num_seqs.png" width="1050" alt="--max-num-seqs가 작으면 넘친 요청이 대기열에서 기다려 TTFT가 길어지고, 크면 모든 요청을 함께 계산해 TPOT가 길어집니다.">

<p style="margin:6px 0;">동시 요청 64개가 한꺼번에 들어오고 상한이 16인 경우를 따라가 봅니다.</p>
<ol style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">16개만 생성 단계에 들어가고 48개는 대기열에서 기다립니다.</li>
<li style="margin:3px 0;">한 단계에 16개만 계산하므로 단계가 짧아 TPOT가 짧습니다.</li>
<li style="margin:3px 0;">앞 요청이 끝나야 자리가 나므로 뒤 요청은 기다린 시간만큼 TTFT가 길어지고, 한 단계에 만드는 토큰이 적어 출력 처리량도 줄어듭니다.</li>
</ol>
<p style="margin:6px 0;">이 교안은 동시 실행 상한을 설명만 하고 측정하지 않습니다. 요청이 몰려도 TPOT를 짧게 지켜야 할 때 상한을 줄여 같은 부하로 측정해 봅니다.</p>
</div>

## 5. 투기적 디코딩: --speculative-config

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">decode는 토큰 하나를 만들 때마다 모델 가중치 전체를 GPU 메모리에서 읽습니다. 그래서 요청이 적으면 계산보다 가중치를 읽는 데 시간이 더 걸려, GPU의 계산 능력이 남습니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">투기적 디코딩(speculative decoding)</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">초안 토큰(draft token) k개를 싸게 만들고, 목표 모델(target model, 서비스하는 모델)을 한 번 계산(순전파)해 k개를 함께 검증하는 생성 방법입니다. 앞에서부터 목표 모델의 답과 맞은 초안 토큰은 수용하고(답에 그대로 씁니다), 처음 틀린 자리에는 목표 모델이 고른 토큰 하나를 더합니다. 초안이 모두 맞으면 그 다음 토큰 하나를 더합니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">목적:</strong> 가중치 읽기 때문에 GPU 계산 능력이 남는 낮은 부하에서, 요청 하나의 생성 속도를 높입니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">수용 규칙:</strong> temperature=0(그리디 생성, 매번 확률이 가장 큰 토큰을 고르는 방식)에서는 목표 모델이 고른 토큰과 같은 초안만 수용합니다. 그래서 남는 토큰은 모두 목표 모델이 고른 토큰입니다. 다만 여러 토큰을 한 번에 검증하는 계산의 수치 차이로 실제 답은 기준 서버와 조금 달라질 수 있습니다.</li>
</ul>
<img style="display:block;max-width:100%;height:auto;margin:0;" src="images/13_speculative_decoding.png" width="1050" alt="목표 모델이 초안 토큰 3개를 순전파 한 번으로 검증해 2개를 수용하고 자기 토큰 1개를 더합니다. 한 단계에 토큰 3개가 나옵니다.">

<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">수용 길이(acceptance length)</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">검증 한 번에 얻는 토큰 수의 평균으로, 1 + 수용된 초안 수의 평균입니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">초안 수용률(draft acceptance rate):</strong> 만든 초안 토큰 가운데 수용된 초안 토큰의 비율입니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">결과 JSON의 키:</strong> 투기적 디코딩을 켠 서버를 벤치마크하면 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">vllm bench serve</code>가 서버의 초안 통계를 읽어, 결과 JSON에 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">spec_decode_acceptance_length</code>(수용 길이)와 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">spec_decode_acceptance_rate</code>(초안 수용률)를 저장합니다. 투기적 디코딩을 켜지 않은 서버의 결과에는 이 키가 없습니다.</li>
</ul>
</div>

<div class="day76-formula" style="margin:10px 0;padding:10px 16px;background:#f3f6fa;border-left:4px solid #1f3a5f;border-radius:4px;font-size:17px;line-height:1.8;color:#202124;"><strong style="color:#1f3a5f;">수용 길이</strong> = 1 + 수용된 초안 토큰 수 ÷ 검증 횟수</div>

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">초안을 2개씩 만들어 검증을 네 번 했고, 수용된 초안이 차례로 2개, 1개, 0개, 2개인 경우를 계산해 봅니다.</p>
<ol style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">수용 길이:</strong> 1 + (2 + 1 + 0 + 2) / 4 = 2.25토큰입니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">초안 수용률:</strong> 만든 초안 8개 가운데 5개라 62.5%입니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">순전파 수:</strong> 검증 네 번에 토큰 9개를 얻었으므로, 목표 모델이 혼자 토큰 9개를 만들 때(순전파 9번)보다 순전파가 다섯 번 적습니다.</li>
</ol>
<p style="margin:6px 0;">추가 모델이 필요 없는 방법은 두 가지입니다. 이 교안은 MTP를 측정합니다.</p>
<table style="background:#ffffff;color:#202124;border-collapse:collapse;margin:6px 0;">
<thead>
<tr>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 8px;">방법</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 8px;">초안을 만드는 곳</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 8px;">잘 맞는 경우</th>
</tr>
</thead>
<tbody>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">MTP(Multi-Token Prediction, 다중 토큰 예측)</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">모델과 함께 학습되어, 다음 토큰 뒤의 토큰까지 미리 예측하는 작은 층(Qwen3.5 허브 모델에 들어 있음)</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">그 모델에 MTP 층이 있을 때</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">n-gram 투기(n-gram speculation)</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">답의 직전 토큰 몇 개(n-gram)와 같은 부분을 입력과 지금까지 만든 답에서 찾아, 그 뒤 토큰</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">답이 입력의 문장을 그대로 옮길 때(요약, 인용)</td>
</tr>
</tbody>
</table>
<p style="margin:6px 0;">설정은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--speculative-config</code>에 JSON 문자열로 넣습니다. MTP는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">{&quot;method&quot;: &quot;mtp&quot;, &quot;num_speculative_tokens&quot;: 2}</code>(초안 2개)입니다. 상담 답은 조문 문장을 그대로 옮기는 부분이 적어 n-gram 초안이 잘 맞지 않을 것으로 보고, 이 교안에서는 n-gram을 측정하지 않습니다.</p>
</div>

## 6. vllm bench sweep serve로 한 번에 측정

### 6-1. vllm bench sweep serve가 하는 일

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">sweep(스윕)</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">설정 값을 하나씩 바꿔 가며 같은 측정을 차례로 반복하는 일입니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">vllm bench sweep serve</code>는 서버 설정마다 서버를 띄우고 같은 벤치마크를 돌리는 일을 명령 하나로 해 줍니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">측정용 서버:</strong> sweep이 설정마다 파드 안(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">127.0.0.1</code>)에서만 띄우는 서버입니다.</li>
</ul>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">vllm bench sweep serve</code>의 순서</strong></p>
<ol style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">서버 설정 파일(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--serve-params</code>)의 설정마다 서버 명령(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--serve-cmd</code>) 뒤에 그 설정의 옵션을 더해 측정용 서버를 띄우고, 준비될 때까지 기다립니다.</li>
<li style="margin:3px 0;">벤치마크 조건 파일(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--bench-params</code>)의 조건마다 벤치마크 명령(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--bench-cmd</code>)에 조건을 더해 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">vllm bench serve</code>를 실행하고 결과 JSON을 저장합니다. 측정이 끝날 때마다 서버의 접두어 캐시를 비워, 앞 측정이 다음 측정에 남지 않게 합니다.</li>
<li style="margin:3px 0;">서버를 끄고 다음 설정으로 넘어갑니다.</li>
<li style="margin:3px 0;">모든 결과를 실험 폴더(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--output-dir</code> 아래 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--experiment-name</code> 폴더)의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">summary.csv</code> 한 파일에 모읍니다. 측정 하나가 한 행입니다.</li>
</ol>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">설정 파일을 옵션으로 바꾸는 규칙</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">키는 옵션 이름의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">-</code>를 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">_</code>로 쓴 것입니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">{&quot;kv_cache_dtype&quot;: &quot;fp8&quot;}</code>는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--kv-cache-dtype fp8</code>이 됩니다.</li>
<li style="margin:3px 0;">값이 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">false</code>이면 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--no-</code> 옵션이 됩니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">{&quot;enable_prefix_caching&quot;: false}</code>는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--no-enable-prefix-caching</code>입니다.</li>
<li style="margin:3px 0;">값이 딕셔너리이면 JSON 문자열로 넣습니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--speculative-config</code>에 쓰는 형식입니다.</li>
<li style="margin:3px 0;">명령에 이미 있는 옵션이면 그 값을 바꿉니다. 벤치마크 조건의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">dataset_path</code>는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--bench-cmd</code>의 입력 파일을 바꿉니다.</li>
<li style="margin:3px 0;">파일을 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">{&quot;이름&quot;: {설정}}</code> 모양으로 쓰면 그 이름이 결과 폴더와 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">summary.csv</code>의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">_benchmark_name</code> 열에 들어갑니다. 두 파일에 모두 이름을 붙이면 이 열에 벤치마크 조건 이름만 남으므로, 이 교안은 서버 설정 파일에만 이름을 붙이고 벤치마크 조건 파일은 이름 없는 리스트로 씁니다.</li>
</ul>
</div>

### 6-2. 비교할 서버 설정과 벤치마크 조건

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">서버 설정은 기준과 5절의 MTP, 둘입니다. 설정 하나마다 서버를 한 번 더 띄워야 하므로 3, 4절의 설정은 측정하지 않습니다. 접두어 캐싱은 두 서버 모두 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--enable-prefix-caching</code>으로 켜 둡니다.</p>
<p style="margin:6px 0;">벤치마크 조건은 셋입니다. 시간을 줄이려고 7절의 비교와 8절 함께 따라하기에 쓰는 조건만 둡니다.</p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">노무, 동시 요청 1(요청 8개):</strong> 낮은 부하입니다. 요청 하나를 빠르게 하는 MTP의 효과를 봅니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">노무, 동시 요청 64(요청 64개):</strong> 높은 부하입니다. GPU 계산이 요청들로 차 있을 때 MTP의 효과가 어떻게 바뀌는지 봅니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">전월세, 동시 요청 16(요청 16개):</strong> 함께 따라하기에서 씁니다. 입력 파일은 교안 01 함께 따라하기에서 올린 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">output/lease_prompts.jsonl</code>입니다.</li>
</ul>
<p style="margin:6px 0;">설정 2개 x 조건 3개로 측정은 6번입니다.</p>
<p style="margin:6px 0;">두 파일은 내 PC의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">output/serve_params.json</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">output/bench_params.json</code>으로 저장해 RunPod에 올립니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">dataset_path</code>는 RunPod 단원 폴더 기준 경로입니다.</p>
</div>

In [ ]:
# 비교할 서버 설정: 이름별로 서버 명령 뒤에 더할 옵션
serve_params = {
    "기준": {},                                                                        # 옵션을 바꾸지 않은 서버
    "MTP": {"speculative_config": {"method": "mtp", "num_speculative_tokens": 2}},     # --speculative-config '{...}'
}

# 서버마다 보낼 벤치마크 조건: 노무는 낮은 부하(동시 요청 1)와 높은 부하(동시 요청 64), 전월세는 동시 요청 16
bench_params = [
    {"dataset_path": "output/labor_prompts.jsonl", "max_concurrency": 1, "num_prompts": 8},
    {"dataset_path": "output/labor_prompts.jsonl", "max_concurrency": 64, "num_prompts": 64},
    {"dataset_path": "output/lease_prompts.jsonl", "max_concurrency": 16, "num_prompts": 16},
]

# sweep이 읽는 JSON 파일로 저장합니다
serve_params_path = output_dir / "serve_params.json"
bench_params_path = output_dir / "bench_params.json"
serve_params_path.write_text(json.dumps(serve_params, ensure_ascii=False, indent=2), encoding="utf-8")
bench_params_path.write_text(json.dumps(bench_params, indent=2), encoding="utf-8")
print("서버 설정:", list(serve_params), "| 벤치마크 조건:", len(bench_params), "개")

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">파일 올리기 · 내 PC → RunPod</strong></p>
<table style="background:#ffffff;color:#202124;border-collapse:collapse;margin:6px 0;">
<thead>
<tr>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 8px;">파일</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 8px;">내 PC 위치</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 8px;">RunPod 위치</th>
</tr>
</thead>
<tbody>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">serve_params.json</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">output/serve_params.json</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">day76_추론가속_서빙벤치마킹/output/serve_params.json</code></td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">bench_params.json</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">output/bench_params.json</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">day76_추론가속_서빙벤치마킹/output/bench_params.json</code></td>
</tr>
</tbody>
</table>
<p style="margin:6px 0;">RunPod JupyterLab 왼쪽 파일 목록에서 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">day76_추론가속_서빙벤치마킹/output</code> 폴더를 열고, 업로드 버튼으로 내 PC의 위 파일을 올립니다.</p>
</div>

### 6-3. RunPod에서 측정 실행

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">2-3에서 공개 서버를 껐는지 확인합니다. 서버가 아직 실행 중이면 RunPod 터미널 1에서 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Ctrl+C</code>로 끄고 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">nvidia-smi</code>로 GPU 메모리가 반환됐는지 확인한 뒤 아래 명령을 실행합니다. sweep이 포트 8000에 측정용 서버를 띄우므로 공개 서버가 떠 있으면 겹칩니다.</p>
</div>

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">명령어 설정</strong></p>
<table style="background:#ffffff;color:#202124;border-collapse:collapse;margin:6px 0;">
<thead>
<tr>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 8px;">설정</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 8px;">의미</th>
</tr>
</thead>
<tbody>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--serve-cmd '...'</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">설정마다 띄울 서버 명령입니다. 1-2의 공개 서버 명령에서 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--api-key</code>와 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">tee</code>를 빼고, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--host 127.0.0.1</code>로 파드 안에서만 요청을 받게 한 모양입니다. 측정용 서버는 밖에서 접속할 필요가 없기 때문입니다</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--bench-cmd '...'</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">측정마다 실행할 벤치마크 명령입니다. 교안 01의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">run_bench</code>에서 동시 요청 수, 요청 수, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--percentile-metrics</code>, 결과 저장 옵션을 뺀 모양입니다. 동시 요청 수와 요청 수는 벤치마크 조건 파일이 붙이고, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--percentile-metrics</code>와 결과 저장은 sweep이 붙입니다</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--serve-params</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--bench-params</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">6-2에서 올린 서버 설정 파일과 벤치마크 조건 파일입니다</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--num-runs 1</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">조건마다 측정 횟수입니다(기본 3). 시간을 줄이려고 한 번만 측정합니다</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--server-ready-timeout 600</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">서버 준비를 기다리는 시간(초, 기본 300)입니다. MTP 서버처럼 준비가 오래 걸리는 설정이 있어 늘립니다</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--output-dir</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--experiment-name</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">결과가 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">output/sweep/labor-settings/</code> 폴더에 쌓입니다</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--show-stdout</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">서버와 벤치마크의 출력을 화면에 보입니다. 주지 않으면 서버 출력이 버려져 시작 로그의 KV 캐시 줄을 확인할 수 없습니다</td>
</tr>
</tbody>
</table>
<p style="margin:6px 0;">화면 출력은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">tee</code>로 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">output/sweep/labor-settings.log</code>에도 남깁니다. 측정용 서버의 API 키는 터미널 준비 블록에서 정한 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">VLLM_API_KEY</code>를 물려받고, 벤치마크는 같은 값의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">OPENAI_API_KEY</code>를 요청에 붙입니다. 터미널 1을 새로 열었다면 1-2의 터미널 1 준비 블록부터 실행합니다.</p>
</div>

**복사해서 실행 · RunPod 터미널 1**

```bash
uv run vllm bench sweep serve \
  --serve-cmd 'vllm serve Qwen/Qwen3.5-4B --served-model-name qwen3.5-4b --max-model-len 8192 --language-model-only --reasoning-parser qwen3 --default-chat-template-kwargs "{\"enable_thinking\": false}" --enable-prompt-tokens-details --enable-prefix-caching --host 127.0.0.1 --port 8000' \
  --bench-cmd 'vllm bench serve --backend openai --base-url http://127.0.0.1:8000 --model qwen3.5-4b --tokenizer Qwen/Qwen3.5-4B --dataset-name custom --dataset-path output/labor_prompts.jsonl --skip-chat-template --custom-output-len 256 --temperature 0 --seed 0 --metric-percentiles 50,95,99 --goodput ttft:1000 tpot:50' \
  --serve-params output/serve_params.json \
  --bench-params output/bench_params.json \
  --num-runs 1 \
  --server-ready-timeout 600 \
  --output-dir output/sweep \
  --experiment-name labor-settings \
  --show-stdout 2>&1 | tee output/sweep/labor-settings.log
```

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">sweep은 서버 시작 두 번과 측정 여섯 번이 이어져 몇 분 넘게 걸립니다. 화면에 서버 시작 로그가 올라오고, 측정마다 벤치마크 결과 블록이 찍히면 정상이므로 끄지 않고 기다립니다. 서버 시작이 한 번에 몇 분씩 걸려, 서버 설정을 하나 더할 때마다 sweep이 그만큼 길어집니다.</p>
<p style="margin:6px 0;">sweep은 uv run 아래에서 실행되므로 안쪽 명령은 vllm으로 바로 시작합니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">다시 실행할 때</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">처음부터 다시:</strong> 같은 실험 이름 폴더가 있으면 sweep이 시작하자마자 멈춥니다. 앞 실험 폴더는 지우지 않고 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">mv output/sweep/labor-settings output/sweep/labor-settings-old</code>로 옮겨 둔 뒤 같은 명령을 다시 실행합니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">이어 하기:</strong> 중단된 실행은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--show-stdout</code> 앞에 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--resume</code>을 붙이고, 마지막 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">tee</code>를 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">tee -a</code>로 바꿔 로그를 이어 씁니다. 결과가 이미 있는 측정은 건너뜁니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">중간에 멈췄을 때:</strong> sweep을 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Ctrl+C</code>로 멈추거나 오류로 끝나면 측정용 서버가 GPU를 잡은 채 남아 있을 수 있습니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">nvidia-smi</code> 아래쪽 프로세스 목록에서 vLLM 프로세스의 PID를 확인해 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">kill &lt;PID&gt;</code>로 끈 뒤 다시 실행합니다.</li>
</ul>
</div>

### 6-4. 설정별 KV 캐시와 블록 크기

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">sweep이 끝나 터미널 1에 다시 명령을 입력할 수 있게 되면(맨 아래 줄에 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">$</code> 표시가 보이면) 아래 명령을 실행합니다.</p>
</div>

**RunPod 터미널 1 · 화면에서 확인**

```bash
grep -E "GPU KV cache size|attention block size" output/sweep/labor-settings.log
```

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">결과 읽기</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">줄 순서:</strong> 서버마다 attention 블록 크기 줄과 KV 캐시 토큰 수 줄이 이 순서로 한 번씩, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">serve_params</code> 순서(기준, MTP)대로 나옵니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">KV 캐시 토큰 수:</strong> GPU 메모리를 같은 비율(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--gpu-memory-utilization</code> 기본값)만큼 써도 MTP 서버는 MTP 층이 메모리를 써 기준보다 적게 담습니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">attention 블록 크기:</strong> 기준 서버의 블록 크기는 2-2 화면의 숫자와 같습니다. MTP 서버의 블록 크기가 기준과 다르면 접두어 캐싱이 다시 쓰는 토큰 수도 달라집니다.</li>
</ul>
</div>

### 6-5. summary.csv 내려받기와 읽기

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">파일 내려받기 · RunPod → 내 PC</strong></p>
<table style="background:#ffffff;color:#202124;border-collapse:collapse;margin:6px 0;">
<thead>
<tr>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 8px;">파일</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 8px;">내 PC 위치</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 8px;">RunPod 위치</th>
</tr>
</thead>
<tbody>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">summary.csv</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">output/sweep/labor-settings/summary.csv</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">day76_추론가속_서빙벤치마킹/output/sweep/labor-settings/summary.csv</code></td>
</tr>
</tbody>
</table>
<p style="margin:6px 0;">RunPod JupyterLab 왼쪽 파일 목록에서 파일을 오른쪽 클릭해 Download를 누릅니다. 내려받은 파일은 내 PC 위치 열의 폴더(단원 폴더 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">day76_추론가속_서빙벤치마킹</code> 기준)에 저장합니다. 브라우저가 Downloads에 저장했다면 이 위치로 옮깁니다.</p>
</div>

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">브라우저가 이름을 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">summary (1).csv</code>처럼 바꿨다면 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">summary.csv</code>로 고쳐 저장합니다.</p>
<p style="margin:6px 0;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">summary.csv</code>에는 결과 JSON의 키(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">output_throughput</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">p95_ttft_ms</code> 등)가 열로, 서버 설정 이름(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">_benchmark_name</code>)과 벤치마크 조건(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">dataset_path</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">max_concurrency</code>)이 함께 들어 있습니다. 첫 열은 이름 없는 행 번호라 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">index_col=0</code>으로 읽습니다.</p>
<p style="margin:6px 0;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">summary.csv</code> 한 행에서 비교 테이블의 한 행을 만드는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">bench_row</code>를 함수로 둡니다. 7절과 함께 따라하기에서 다시 씁니다. 목표 달성률(attainment)은 교안 01과 같이 goodput을 요청 처리량으로 나눈 값입니다.</p>
</div>

In [ ]:
def bench_row(setting, concurrency, result):
    """결과 JSON 딕셔너리나 summary.csv의 한 행에서 비교 테이블의 한 행을 만듭니다."""
    return {
        "설정": setting,
        "동시 요청 수": concurrency,
        "출력 처리량(토큰/초)": result["output_throughput"],
        "p95 TTFT(ms)": result["p95_ttft_ms"],
        "p95 TPOT(ms)": result["p95_tpot_ms"],
        "goodput(요청/초)": result["request_goodput"],
        "평균 TTFT(ms)": result["mean_ttft_ms"],
        "평균 TPOT(ms)": result["mean_tpot_ms"],
        "목표 달성률": result["request_goodput"] / result["request_throughput"],  # 두 지연 목표를 모두 지킨 요청의 비율
    }


# 모든 비교 테이블에서 함께 보는 열
compare_columns = ["설정", "동시 요청 수", "출력 처리량(토큰/초)", "p95 TTFT(ms)", "p95 TPOT(ms)", "goodput(요청/초)"]

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">노무 입력 파일의 행만 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">sweep_table</code>에 담고, 전월세 입력 파일의 행은 8절 함께 따라하기에서 씁니다.</p>
</div>

In [ ]:
# 실험 폴더의 summary.csv: 측정 하나가 한 행입니다(첫 열은 행 번호)
sweep_results = pd.read_csv(sweep_dir / "labor-settings" / "summary.csv", index_col=0)

# 노무 입력 파일의 행마다 비교 테이블의 한 행을 만듭니다. 수용 길이는 투기적 디코딩 서버의 행에만 값이 있고 나머지는 빈칸(NaN)입니다
sweep_rows = []
for result in sweep_results.to_dict("records"):
    if result["dataset_path"] == "output/labor_prompts.jsonl":
        row = bench_row(result["_benchmark_name"], result["max_concurrency"], result)
        row["수용 길이"] = result["spec_decode_acceptance_length"]
        sweep_rows.append(row)

sweep_table = pd.DataFrame(sweep_rows).round(2)
display(sweep_table)

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">결과 읽기</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">노무 4행:</strong> 설정 2개 x 동시 요청 수 2개(1, 64)가 모두 있습니다. 설정 이름은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">serve_params</code>의 키 그대로입니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">수용 길이:</strong> MTP 행에만 값이 있습니다.</li>
</ul>
</div>

## 7. 기준과 MTP 비교, 서버 설정 정하기

### 7-1. 투기적 디코딩: 낮은 부하와 높은 부하의 기준과 MTP

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">6-5의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">sweep_table</code>에서 같은 동시 요청 수끼리 기준 행과 MTP 행을 비교합니다. 수용 길이는 MTP 행에만 있습니다.</p>
</div>

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">결과 읽기</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">수용 길이:</strong> 초안 2개를 검증하므로 1 이상 3 이하입니다. 1이면 초안이 한 번도 맞지 않은 것입니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">동시 요청 1:</strong> 평균 TPOT가 짧아지고 출력 처리량이 늘어나는지 비교합니다. 초안을 만들고 검증하는 계산이 순전파마다 더해지므로, 빨라지는 배수는 수용 길이보다 작습니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">동시 요청 64:</strong> GPU 계산이 요청들로 이미 차 있으면 검증 계산이 출력 처리량을 깎고, MTP 층이 메모리를 써 KV 캐시도 줄어듭니다(6-4 화면). sweep 로그에서 KV 캐시 사용률이 거의 100%까지 차고 Waiting이 생기는 것으로 확인할 수 있습니다. 동시 요청 1과 달리 기준과의 차이가 줄거나 손해로 바뀌는지 비교합니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">TTFT:</strong> 측정 예(L4)에서는 두 부하 모두 MTP의 TTFT가 기준보다 길었습니다. 목표 달성률은 TTFT 목표로도 떨어지므로, 동시 요청 1에서 TPOT와 출력 처리량이 앞서도 7-2와 8-2에서 MTP가 골라지지 않을 수 있습니다.</li>
</ul>
</div>

### 7-2. 서버 설정 정하기

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">서비스의 예상 동시 요청 수에서 두 설정을 비교해 하나를 고릅니다. 고르는 규칙은 교안 01에서 동시 요청 수를 고를 때와 같습니다. 그 동시 요청 수에서 목표 달성률이 0.95 이상인 설정 가운데 출력 처리량이 가장 큰 설정을 고릅니다.</p>
</div>

In [ ]:
expected_concurrency = 1  # 서비스의 예상 동시 요청 수(sweep의 노무 조건 1, 64 가운데 하나). 64로 바꿔 다시 실행하면 어떤 설정이 남는지(하나도 남지 않을 수도 있습니다) 비교할 수 있습니다

# 반올림 전 값인 sweep_rows에서 예상 동시 요청 수의 행만 보고 고릅니다(테이블은 둘째 자리에서 반올림한 값이라 반올림 전 값으로 비교합니다)
best_setting = None
best_throughput = 0
for row in sweep_rows:
    if row["동시 요청 수"] == expected_concurrency and row["목표 달성률"] >= 0.95 and row["출력 처리량(토큰/초)"] > best_throughput:
        best_setting = row["설정"]
        best_throughput = row["출력 처리량(토큰/초)"]

# 결정 근거 테이블: 예상 동시 요청 수의 두 설정
decision_table = sweep_table[sweep_table["동시 요청 수"] == expected_concurrency]
display(decision_table[["설정", "출력 처리량(토큰/초)", "p95 TTFT(ms)", "p95 TPOT(ms)", "목표 달성률"]])

# 고른 설정이 없으면 이 동시 요청 수에서는 어떤 설정도 지연 목표를 95% 이상 지키지 못한 것입니다
if best_setting is None:
    print(f"동시 요청 {expected_concurrency}에서 지연 목표를 95% 이상 지킨 설정이 없습니다. 지연 목표를 다시 정하거나, 서버 한 대에 보내는 동시 요청 수를 줄이거나 서버를 늘립니다.")
else:
    print(f"동시 요청 {expected_concurrency}에서 고른 설정: {best_setting} | 더할 옵션: {serve_params[best_setting]} | 출력 처리량: {best_throughput:.1f} 토큰/초")

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">결과 읽기</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">best_setting</code>:</strong> 결정 근거 테이블에서 목표 달성률이 0.95 이상인 행 가운데 출력 처리량이 가장 큰 설정입니다. 그런 행이 없으면 고른 설정 대신 안내 문장이 나옵니다. 예상 동시 요청 수를 64로 바꾸면 지연 목표를 지키는 설정이 줄어듭니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">동시 요청 1:</strong> 요청이 8개라 한 요청만 지연 목표를 넘어도 목표 달성률이 0.875(8개 중 7개)가 되어 탈락합니다. sweep은 측정마다 접두어 캐시를 비우므로 측정의 첫 요청은 시스템 메시지 전체를 prefill 해 TTFT가 길어집니다. 그래서 MTP의 TPOT가 더 짧아도 이런 요청 하나 때문에 기준이 골라지거나, 두 설정 모두 탈락할 수 있습니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">실무:</strong> 고른 설정의 옵션을 6-1의 규칙대로 서버 명령 뒤에 붙여 서버를 띄우고, 서버 명령은 결정 근거 테이블(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">decision_table</code>)과 함께 남깁니다. 같은 설정도 실행마다 출력 처리량이 조금씩 달라지므로, 차이가 작으면 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--num-runs</code>를 2~3으로 늘려 다시 측정합니다.</li>
</ul>
</div>

## 8. 🖐️ 함께 따라하기: 전월세 창구의 서버 설정 정하기

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">전월세 상담 창구의 예상 동시 요청 수에서 지연 목표를 지키는 서버 설정을, 같은 sweep 결과에서 기준과 MTP 가운데 고릅니다. 서버를 새로 띄우지 않습니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">본실습과 다른 점</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">데이터:</strong> 같은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">summary.csv</code>의 전월세 조건 행입니다. 입력 파일은 교안 01 함께 따라하기에서 올린 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">output/lease_prompts.jsonl</code>입니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">예상 동시 요청 수:</strong> 16입니다(전월세 조건은 동시 요청 16, 요청 16개).</li>
</ul>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">실습 진행 방법:</strong> 작성 셀의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">...</code>만 주석에 맞춰 채웁니다. 본실습의 변수와 함수(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">sweep_results</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">serve_params</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">bench_row</code>)를 그대로 쓰므로 본실습을 먼저 실행합니다. 완성 코드는 수업이 끝난 뒤 강사가 공유합니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">설정:</strong> 서버 설정 두 개(기준, MTP)와 벤치마크 옵션(생성 상한 256토큰, 지연 목표 TTFT 1000ms, TPOT 50ms)은 본실습과 같습니다.</p>
</div>

### 8-1. 전월세 행 고르기

<details><summary>힌트</summary>

```text
접근방법: 교안 02의 6-5를 참고합니다.
summary.csv의 dataset_path 열 값이 6-2의 bench_params에 적은 전월세 입력 파일 경로와 같은 행만 고릅니다.
```

</details>

In [ ]:
# 함께 따라하기: dataset_path가 전월세 입력 파일인 행만 고르는 조건을 채우세요
lease_sweep_rows = []
for result in sweep_results.to_dict("records"):
    if ...:
        row = bench_row(result["_benchmark_name"], result["max_concurrency"], result)
        row["수용 길이"] = result["spec_decode_acceptance_length"]
        lease_sweep_rows.append(row)

lease_sweep_table = pd.DataFrame(lease_sweep_rows).round(2)
display(lease_sweep_table)

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">확인 기준</strong>: <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">lease_sweep_table</code>이 두 행(기준, MTP)이고, 동시 요청 수가 모두 16입니다.</p>
</div>

### 8-2. 전월세 창구의 설정 고르기

<details><summary>힌트</summary>

```text
접근방법: 교안 02의 7-2를 참고합니다.
전월세 행은 모두 동시 요청 16이므로 목표 달성률과 지금까지의 최대 출력 처리량, 두 조건만 비교합니다.
```

</details>

In [ ]:
# 함께 따라하기: 7-2와 같은 규칙(목표 달성률 0.95 이상, 출력 처리량 최대)의 두 조건을 채우세요
lease_best_setting = None
lease_best_throughput = 0
for row in lease_sweep_rows:
    if ... and ...:
        lease_best_setting = row["설정"]
        lease_best_throughput = row["출력 처리량(토큰/초)"]

# 고른 설정이 없으면 동시 요청 16에서 어떤 설정도 지연 목표를 95% 이상 지키지 못한 것입니다
if lease_best_setting is None:
    print("동시 요청 16에서 지연 목표를 95% 이상 지킨 설정이 없습니다. 서버 한 대에 보내는 동시 요청 수를 줄이거나 서버를 늘립니다.")
else:
    print(f"전월세 창구의 설정: {lease_best_setting} | 더할 옵션: {serve_params[lease_best_setting]} | 출력 처리량: {lease_best_throughput:.1f} 토큰/초")

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">출력 처리량이 더 큰 설정도 목표 달성률이 0.95 미만이면 고르지 않습니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">확인 기준</strong>: 고른 설정은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">lease_sweep_table</code>에서 목표 달성률이 0.95 이상인 행 가운데 출력 처리량이 가장 큰 행의 설정과 같습니다. 0.95 이상인 행이 없을 때만 안내 문장이 나옵니다.</p>
<p style="margin:6px 0;">sweep은 서버 설정마다 측정을 마치면 그 서버를 끄므로, 마지막 MTP 서버도 sweep이 끝날 때 꺼져 있습니다. RunPod 터미널 1에서 GPU 메모리가 반환됐는지 확인합니다.</p>
</div>

**RunPod 터미널 1 · 화면에서 확인**

```bash
nvidia-smi
```

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">확인 기준</strong>: vLLM 서버 프로세스가 없고 GPU 메모리 사용량이 2-3에서 공개 서버를 끈 뒤와 같은 수준입니다.</p>
</div>

## 9. 핵심 정리

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">접두어 캐싱(<code style="color:inherit;background:#f3f6fa;font-size:0.95em;">--enable-prefix-caching</code>):</strong> vLLM 0.27.0은 Qwen3.5 같은 하이브리드 모델에서 기본으로 꺼 두므로 직접 켭니다. 앞에서부터 같은 KV 캐시 블록의 prefill을 건너뛰어 TTFT와 GPU 시간을 줄이고, 블록 크기 단위로만 적중합니다. 공통 부분(지침, 참고 조문)을 앞에, 요청마다 다른 부분(질문)을 뒤에 둡니다. 질문을 앞에 두면 다시 쓸 블록이 없어 캐싱을 끈 서버와 같은 prefill을 합니다. 측정 예(L4, 동시 요청 16)에서는 캐싱을 켜면 같은 요청 64개를 약 2.5배 빨리 끝냈습니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">KV 캐시 자료형(<code style="color:inherit;background:#f3f6fa;font-size:0.95em;">--kv-cache-dtype fp8</code>):</strong> KV 캐시 값을 1바이트로 저장해 같은 메모리에 담는 토큰을 늘립니다. Qwen3.5는 attention 층의 KV만 줄어 2배보다 적게 늘고, attention 블록이 커져 캐시 적중이 줄면 TTFT가 늘 수 있습니다. 이 교안은 측정하지 않았고, 동시 요청이 많아 KV 캐시가 모자랄 때 측정해 봅니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">동시 실행 상한(<code style="color:inherit;background:#f3f6fa;font-size:0.95em;">--max-num-seqs</code>):</strong> 한 생성 단계에 함께 계산하는 요청 수를 줄여 TPOT를 짧게 합니다. 대가는 대기열에서 기다린 시간이 더해진 TTFT와 줄어든 출력 처리량입니다. 이 교안은 측정하지 않았고, 요청이 몰려도 TPOT를 짧게 지켜야 할 때 측정해 봅니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">투기적 디코딩(<code style="color:inherit;background:#f3f6fa;font-size:0.95em;">--speculative-config</code>):</strong> 초안 토큰을 목표 모델이 한 번에 검증해 순전파 한 번에 수용 길이만큼 토큰을 얻습니다. 초안을 만들고 검증하는 계산이 대가라, 수용 길이가 1에 가깝거나 동시 요청이 많으면 출력 처리량에서 손해일 수 있습니다. 측정 예에서는 모든 조건에서 TTFT가 길어져, 낮은 부하에서 TPOT와 출력 처리량이 앞서도 지연 목표로 고르면 기준에 질 수 있습니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">vllm bench sweep serve</code>:</strong> 서버 설정 파일과 벤치마크 조건 파일의 조합마다 서버를 띄우고 측정하고 끄기를 명령 하나로 반복하고, 결과를 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">summary.csv</code>에 모읍니다. 설정 키는 옵션 이름의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">-</code>를 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">_</code>로 쓰고, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">false</code>는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--no-</code> 옵션, 딕셔너리는 JSON 문자열이 됩니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">서버 설정 정하기:</strong> 예상 동시 요청 수에서 기준과 MTP를 비교해, 목표 달성률 0.95 이상인 설정 가운데 출력 처리량이 가장 큰 설정을 고르고, 그 옵션과 결정 근거 테이블을 함께 남깁니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">실행 위치:</strong> 서버, sweep, 로그 확인은 RunPod 터미널에서, 요청 하나, 설정 파일 만들기, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">summary.csv</code> 분석은 내 PC 노트북에서 합니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">서버 시작 줄이기:</strong> 서버 시작은 느리므로 같은 설정의 서버는 이어 쓰고, 설정을 바꿀 때만 sweep으로 몰아서 띄웁니다.</li>
</ul>
</div>